# Config

In [2]:
import os

# Ruta a la que quieres mover el path
nueva_ruta = "/tmp/Repository/VRID_language_proyect/BERT"

# Cambiar el directorio actual
os.chdir(nueva_ruta)

# Confirmar que cambió
print("Directorio actual:", os.getcwd())

Directorio actual: /tmp/Repository/VRID_language_proyect/BERT


In [3]:
import pandas as pd
import os
from preprocess.preprocess import clean_text, check_deleted_expressions
from preprocess.translate import translator, gen_text_for_embedding, final_clean
import time

# 1) Preprocesamiento de los datos


In [ ]:
# 1) Cargar datos
path = "/tmp/data"
path_analytics = "/tmp/analytics"
filePATH = os.path.join(path, "data_concatenada.xlsx")
df = pd.read_excel(filePATH,
                   usecols=["Código VRID", "Título", "Resumen", "Keywords", "Interdisciplinario", "Transdisciplinario", "Facultad del Proyecto",
                            "Depto Persona"]) \
       .fillna("")

# 2) Guardar qué secuencias de palabras del resumen serán eliminadas al aplicar get_expressions_to_delete()
list_texts = df["Resumen"].to_list()
df_deleted = check_deleted_expressions(list_texts)
savepath=os.path.join(path_analytics, "deleted_re.xlsx")
df_deleted.to_excel(savepath, index=False)

# 3) Preprocesar los datos
#Columnas que se van a preprocesar
#Nombre fila seleccionada:Columna que se creará para guardar resultado
cols = {
    "Título": "Titulo_trad",
    "Resumen": "Resumen_trad",
    "Keywords": "keywords_trad",
    "Facultad del Proyecto": "Facultad_del_Proyecto_trad",
    "Depto Persona": "Depto_Persona_trad",
}
#Preprocesamiento de datos
df[list(cols.values())] = df[list(cols.keys())].applymap(clean_text)
savepath=os.path.join(path, "data_clean.xlsx")
df.to_excel(savepath, index=False)

# 2) Traducción del texto

In [10]:
from transformers import MarianMTModel, MarianTokenizer

#1. Cargar modelo de traducción
model_name = "Helsinki-NLP/opus-mt-es-en"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)
trans = translator(model, tokenizer)

#Columnas que se van a traducir
#Nombre fila seleccionada:Columna que se creará para guardar resultado
cols = {
    "Titulo_trad": "Titulo_trad",
    "Resumen_trad": "Resumen_trad",
    "keywords_trad": "keywords_trad",
    "Facultad_del_Proyecto_trad": "Facultad_del_Proyecto_trad",
    "Depto_Persona_trad": "Depto_Persona_trad",
}

#2. Traducción de columnas
#####Estoy trabajando en mejorar esta parte para que sea más rápida con paralelización por batches
start = time.time()
for src, dst in cols.items():
    df[dst] = trans.translate_parallel(df[src].to_list(), batch_size=8)
end = time.time()


#3.Guardado de resultados
savepath=os.path.join(path, "data_translated.xlsx")
df.to_excel(savepath, index=False)

print(f"Tiempo total de traducción: {end - start:.2f} segundos")

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/826k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Usando dispositivo: cuda


model.safetensors:   0%|          | 0.00/312M [00:00<?, ?B/s]

Traduciendo: 100%|██████████| 119/119 [00:27<00:00,  4.40batch/s]


Tiempo total de traducción: 769.17 segundos


In [11]:
#3. Selección de columnas que se utilizarán en clasificador y concatenación
# Última limpieza antes de generar concatenación
cols = ["Titulo_trad", "keywords_trad", "Resumen_trad", "Facultad_del_Proyecto_trad", "Depto_Persona_trad"]
for col in cols:
    df[col] = df[col].apply(final_clean)

#  Selección de columnas para embedding.
cols = ["Titulo_trad", "keywords_trad", "Resumen_trad"]
element_names = ["title", "keywords", "abstract"]
df = gen_text_for_embedding(df, cols, element_names)

# Guardado de resultados
savepath=os.path.join(path, "data_translated_concat.xlsx")
df.to_excel(savepath, index=False)
savepath=os.path.join(path, "data_translated_concat.csv")
df.to_csv(savepath, index=False, encoding="utf-8-sig")

df.head()

,Código VRID,Interdisciplinario,Transdisciplinario,Título,Keywords,Resumen,Facultad del Proyecto,Depto Persona,Titulo_trad,Resumen_trad,keywords_trad,Facultad_del_Proyecto_trad,Depto_Persona_trad,text_for_embedding_translated
0,217.173.049-1.0,SI,,PATRONES DE CRIANZA Y SOCIALIZACIÓN DE GÉNERO ...,,OBJETIVOS GENERALES: _x000D_\nDESCRIBIR LOS PR...,FACULTAD DE CIENCIAS SOCIALES,"DEPARTAMENTO DE ECONOMÍA, SIN INFORMACIÓN, ESC...",patrones de crianza y socialización de género ...,objetivos generales: describir los procesos de...,,facultad de ciencias sociales,"departamento de economía, sin información, esc...",title: patrones de crianza y socialización de ...
1,218.201.002-1.0,SI,,ADAPTACIÓN CULTURAL Y VALIDACIÓN DE LA ESCALA ...,"ESTILO DE VIDA, ADOLESCENTES _x000D_\n",PARA EVALUAR LOS COMPORTAMIENTOS RELACIONADOS ...,FACULTAD DE ENFERMERÍA,"DEPARTAMENTO DE CIENCIA ANIMAL, DEPARTAMENTO D...",adaptación cultural y validación de la escala ...,para evaluar los comportamientos relacionados ...,"estilo de vida, adolescentes",facultad de enfermería,"departamento de ciencia animal, departamento d...",title: adaptación cultural y validación de la ...
2,218.102.031-1.0IN,NO,,PROMOVIENDO LA REFLEXIÓN EN ESTUDIANTES DE PRE...,,EL PRESENTE PROYECTO INVOLUCRA LA REALIZACIÓN ...,FACULTAD DE ODONTOLOGÍA,DEPARTAMENTO DE ASTRONOMÍA,promoviendo la reflexión en estudiantes de pre...,el presente proyecto involucra la realización ...,,facultad de odontología,departamento de astronomía,title: promoviendo la reflexión en estudiantes...
3,218.163.016-INI,INDEFINIDO,,MOTIVACIÓN Y HABILIDADES SOCIALES EN ADOLESCENTES,,EL ESTUDIO DE LA MOTIVACIÓN TIENE DIFERENTES A...,FACULTAD DE EDUCACIÓN,"DEPARTAMENTO DE CIENCIAS DE LA EDUCACIÓN, DEPT...",motivación y habilidades sociales en adolescentes,el estudio de la motivación tiene diferentes a...,,facultad de educación,"departamento de ciencias de la educación, dept...",title: motivación y habilidades sociales en ad...
4,219.091.052-INI,NO,,TIME EFFECTS ON THE LIQUEFACTION RESPONSE OF G...,,SECONDARY CONSOLIDATION AND AGEING ARE TWO OFT...,FACULTAD DE INGENIERÍA,DEPTO. TEORÍA POLITICA Y FUND.DE LA EDUC.,time effects on the liquefaction response of g...,secondary consolidation and ageing are two oft...,,facultad de ingeniería,depto. teoría politica y fund.de la educ.,title: time effects on the liquefaction respon...


# 3) Split dataset

In [56]:
def to_serializable(obj):
    if hasattr(obj, "tolist"):
        return obj.tolist()
    return obj

In [57]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold
import json

path = "/tmp/data"
filepath=os.path.join(path, "data_translated_concat.csv")
df = pd.read_csv(filepath)

df = df[df["Interdisciplinario"] != "INDEFINIDO"]
le = LabelEncoder()
df["labels"] = le.fit_transform(df["Interdisciplinario"])

# Convertir a arrays
ids = df["Código VRID"].to_numpy()
labels = df["labels"].to_numpy()

# Train/Test split (ids y labels en paralelo)
idx_train, idx_test, y_train, y_test = train_test_split(
    ids,
    labels,
    test_size=0.2,
    random_state=7,
    stratify=labels
)

# Crear folds sobre train
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=7)

folds = []
for fold, (train_pos, val_pos) in enumerate(skf.split(idx_train, y_train)):
    train_ids = idx_train[train_pos]   # array de IDs
    val_ids = idx_train[val_pos]       # array de IDs
    folds.append({
        "fold": fold,
        "train_idx": train_ids,
        "val_idx": val_ids
    })

print("Test size:", len(idx_test))
print("Fold 0 - Train size:", len(folds[0]["train_idx"]))
print("Fold 0 - Val size:", len(folds[0]["val_idx"]))

#Guardar index en diccionario
dataset_index = {
    "Train": idx_train,
    "Test": idx_test,
    "kfolds": folds 
}
filepath=os.path.join(path, "train_test_ids_3folds.json")

# Guardar
with open(filepath, "w", encoding="utf-8") as f:
    json.dump(dataset_index, f, default=to_serializable, indent=2, ensure_ascii=False)


Test size: 193
Fold 0 - Train size: 514
Fold 0 - Val size: 257


# 4) TF-ID feature extractor 

In [62]:
#Lectura de index de separacion de conjuntos train/test
path = "/tmp/data"
filepath=os.path.join(path, "train_test_ids_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
path = "/tmp/data"
filepath=os.path.join(path, "data_translated_concat.csv")
df = pd.read_csv(filepath)

In [52]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.linear_model import SGDClassifier

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, make_scorer

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score
)

import copy
from sklearn.model_selection import train_test_split

from sklearn.datasets import make_classification
from sklearn.metrics import confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight

from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek
from imblearn.under_sampling import RandomUnderSampler
from collections import Counter

from sklearn.decomposition import PCA

from skopt import BayesSearchCV
from skopt.space import Real, Categorical, Integer
import numpy as np

#import mlflow

def eval_model(best_model, X_test, y_test):
  results = {}
  preds = best_model.predict(X_test)
  cm = confusion_matrix(y_test, preds)
  t_n, f_p, f_n, t_p = cm.ravel()
  results = {
      'accuracy': accuracy_score(y_test, preds),
      'precision': precision_score(y_test, preds, zero_division=0),
      'recall': recall_score(y_test, preds, zero_division=0),
      'f1_score': f1_score(y_test, preds, zero_division=0),
      't_n': t_n,
      'f_p': f_p,
      'f_n': f_n,
      't_p': t_p
  }
  return results

#MLflow logging helper function
def safe_log_metric(name, value):
    try:
        if isinstance(value, (list, tuple, np.ndarray)):
            if np.size(value) == 1:
                value = float(np.array(value).item())
            else:
                raise ValueError("Métrica con más de un valor.")
        else:
            value = float(value)
        mlflow.log_metric(name, value)
    except Exception as e:
        print(f"⚠️ No se pudo loggear {name}: {e}")

def mlflow_ckeckpoint(results_val, models_dicc, X_test, y_test, experiment_name):
    # Define el experimento (lo crea si no existe) 
    mlflow.set_tracking_uri("sqlite:///mlflow.db")
    mlflow.set_experiment(experiment_name)

    for model_name, metrics in results_val.items():
        model = models_dicc[model_name]

        with mlflow.start_run(run_name=model_name):
            print(f"Registrando modelo en MLflow: {model_name}")

            # Hiperparámetros
            try:
                mlflow.log_params(model.get_params())
            except:
                print(f"No se pudieron loggear los hiperparámetros para {model_name}")

            # Métricas de validación
            for k, v in metrics.items():
                safe_log_metric(f"val_{k}", v)

            # Métricas de test
            results_test = eval_model(model, X_test, y_test)
            for k, v in results_test.items():
                safe_log_metric(f"test_{k}", v)

            # Guardar modelo
            mlflow.sklearn.log_model(model, name = "model", input_example=X_test[:5])  

#Pipeline helper functions
def get_est_params_dict(keys):
    clf_params_dict = {
        'LogisticRegression': {
            'class': LogisticRegression,
            'params': {
                'solver': Categorical(['lbfgs']),
                'penalty': Categorical(['l2']),
                'C': [0.1, 1.0, 10]
            }
        },
        'RandomForestClassifier': {
            'class': RandomForestClassifier,
            'params': {
                'bootstrap': Categorical([True]),
                'n_estimators': Integer(40, 300),
                'max_depth': Integer(3, 14),
                'min_samples_split': Integer(2, 5),
                'min_samples_leaf': Integer(1, 2),
            }
        },
        'GradientBoostingClassifier': {
            'class': GradientBoostingClassifier,
            'params': {
                'n_estimators': Integer(50, 200),
                'learning_rate': Real(0.01, 0.2),
                'max_depth': Integer(3, 7)
            }
        },
        'XGBClassifier': {
            'class': XGBClassifier,
            'params': {
                'n_estimators': Integer(40, 300),
                'learning_rate': Real(0.01, 0.2),
                'max_depth': Integer(3, 14),
                'eval_metric': Categorical(['logloss'])
            }
        },

    }

    # Filtrar y retornar solo los modelos solicitados
    return {key: clf_params_dict[key] for key in keys if key in clf_params_dict}

def setup_model(dicc):
    model_ = dicc['class']()
    #scaler = StandardScaler()
    model = Pipeline([('model', model_)])
    param_grid = {'model__' + param_name: param_value for param_name, param_value in dicc['params'].items()}
    return model, param_grid

def run_BayesSearchCV(model, param_grid, X_train, y_train, n_iter=10, scoring='recall', sample_weight=None):
    from sklearn.exceptions import FitFailedWarning
    import warnings

    bayes_searchCV = BayesSearchCV(
        estimator=model,
        search_spaces=param_grid,
        cv=5,
        scoring=scoring,
        n_iter = n_iter,
        n_jobs=4,
        n_points = 2
    )

    try:
        fit_params = {'model__sample_weight': sample_weight} if sample_weight is not None else {}
        bayes_searchCV.fit(X_train, y_train, **fit_params)
    except TypeError as e:
        print(f"⚠️ Modelo {model.named_steps['model'].__class__.__name__} no acepta sample_weight. Reintentando sin él.")
        bayes_searchCV.fit(X_train, y_train)

    return bayes_searchCV

def calculate_metrics(y_true, y_pred, y_proba=None):
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1_score": f1_score(y_true, y_pred, zero_division=0),
    }

    # AUROC y AUPRC requieren probabilidades (y_proba)
    if y_proba is not None:
        try:
            metrics["auroc"] = roc_auc_score(y_true, y_proba)
            metrics["auprc"] = average_precision_score(y_true, y_proba)
        except:
            metrics["auroc"] = np.nan
            metrics["auprc"] = np.nan

    return metrics

def get_sample_weight(y_train):
    """
    Devuelve un array de sample weights donde las muestras con y=1
    tienen 10 veces más peso que las muestras con y=0.
    """
    y_train = np.asarray(y_train)
    weights = np.ones_like(y_train, dtype=np.float64)
    weights[y_train == 1] = 10
    return weights

def select_best_model(results_val, models_dicc):
    best_result = 0
    best_model = None
    for model, metrics in results_val.items():
        if metrics['mean_test_score'] > best_result:
            best_result = metrics['mean_test_score']
            best_model = models_dicc[model]
    return best_model

#Pipeline function to run the entire ML pipeline
def run_bayesian_pipeline(est_params_dict, data, labels, n_iter, sample_weight_On = None):

    # creacion de diccionarios para almacenamiento
    results_test = {}
    cm_test = {}
    results_val = {}
    models_dicc = {}

    for model_name, dicc in est_params_dict.items():

        print(model_name)
        model, param_grid = setup_model(dicc)

        #Sample weights strategy
        if sample_weight_On is not None:
          print('Compute sw')
          sample_weight = compute_sample_weight(class_weight='balanced', y=labels)
        else:
          sample_weight = None
        grid_search = run_BayesSearchCV(model, param_grid, data, labels, n_iter = n_iter, scoring = 'recall', sample_weight = sample_weight)


        # Guardar best model
        best_model = grid_search.best_estimator_
        models_dicc[model_name] = copy.deepcopy(best_model)

        mean_val_score = grid_search.cv_results_['mean_test_score'][grid_search.best_index_]
        std_val_score = grid_search.cv_results_['std_test_score'][grid_search.best_index_]
        mean_val_score = np.round(mean_val_score,2)
        std_val_score = np.round(std_val_score,2)

        metrics_val = {
          'mean_test_score': float(np.round(mean_val_score, 2)),
          'std_test_score': float(np.round(std_val_score, 2)
        )}

        results_val[model_name]  = metrics_val


    return results_val, models_dicc


# Ejemplo de uso

# 1. Generar dataset sintético
X, y = make_classification(
    n_samples=1000,
    n_features=20,
    n_informative=10,
    n_redundant=5,
    n_repeated=0,
    n_classes=2,
    #weights=[0.95],  # 90% clase 0, 10% clase 1
    random_state=42,
    shuffle=True
)

# 2. Elegir modelos a probar
model_keys = [
    #'LogisticRegression',
    #'DecisionTreeClassifier',
    #'RandomForestClassifier',
    #'GradientBoostingClassifier',
    'XGBClassifier',
    #'MLPClassifier',
    #'SVC',
    #'SGDClassifier'
]

# 3. Obtener el diccionario de modelos y parámetros
est_params_dict = get_est_params_dict(model_keys)

print("📊 Comienzo:", Counter(y))
# 4. Ejecutar entrenamiento, validación y test con tus funciones
results_val, models_dicc = run_bayesian_pipeline(est_params_dict, X, y, n_iter=20, sample_weight_On = True)

best_model = select_best_model(results_val, models_dicc)

# 5. Mostrar resultados
print("\n🔍 Validación:")
for model, metrics in results_val.items():
    print(f"{model}: {metrics}")


📊 Comienzo: Counter({1: 503, 0: 497})
XGBClassifier
Compute sw

🔍 Validación:
XGBClassifier: {'mean_test_score': 0.94, 'std_test_score': 0.03}
